In [1]:
# Data Cleaning

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv(r"D:\Healthcare Data Analytics Project\data\diabetic_data.csv")

In [4]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [14]:
df.shape

(101766, 50)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [7]:
df.isnull().sum().sum()

0

In [12]:
# There are no null/NaN values present in the dataframe

In [8]:
df.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='object')

In [16]:
# From the above list of columns, we need to select columns that determines readmission after 30 days of discharge as per the aim of the task:
# The target variable is the 'readmitted' column in this case.
# Other variables that determine/predicts the readmission are: 
# Age- It is the strongest predictor of recovery and risk
# Time in hospital - Direct link to severity and illness
# num_lab_procedure - Represents the intensity of the diagnostic process
# num_medications - Indicates "Polypharmacy". High risk if many drugs are consumed
# number_inpatient - Past history is a strong predictor of future readmissions
# diag_1, diag_2, doag_3 - Primary/Secondary reasons for stay
# A1Cresult - Crucial medicine for sugar control
# insulin - High risk patients are on insulin management
# diabetesMed - Tells if diabetes medication was prescribed

# race
# gender
# admission_type_id
# discharge_disposition_id
# number_emergency
# number_outpatient
# change

# common_meds = ['metformin', 'insulin', 'glipizide', 'glyburide']

# summary_meds = ['change', 'diabetesMed']

In [11]:
# Instead of tracking 23 obscure drugs, our AI focuses on Insulin stability and A1C results, which are the primary drivers of readmission risk.

In [15]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [17]:
# Lets keep the relevant columns and remove the non-neccessary ones

selected_columns = [
    'race', 'gender', 'age', 'time_in_hospital', 'num_lab_procedures', 
    'num_procedures', 'num_medications', 'number_outpatient', 
    'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 
    'number_diagnoses', 'A1Cresult', 'metformin', 'glipizide', 
    'glyburide', 'insulin', 'change', 'diabetesMed', 'readmitted', 'admission_type_id', 'discharge_disposition_id'
]

In [21]:
df = df[selected_columns].copy()

In [22]:
df.shape

(101766, 24)

In [26]:
df.head()

,race,gender,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,...,A1Cresult,metformin,glipizide,glyburide,insulin,change,diabetesMed,readmitted,admission_type_id,discharge_disposition_id
0,Caucasian,Female,[0-10),1,41,0,1,0,0,0,...,None,No,No,No,No,No,No,NO,6,25
1,Caucasian,Female,[10-20),3,59,0,18,0,0,0,...,None,No,No,No,Up,Ch,Yes,>30,1,1
2,AfricanAmerican,Female,[20-30),2,11,5,13,2,0,1,...,None,No,Steady,No,No,No,Yes,NO,1,1
3,Caucasian,Male,[30-40),2,44,1,16,0,0,0,...,None,No,No,No,Up,Ch,Yes,NO,1,1
4,Caucasian,Male,[40-50),1,51,0,8,0,0,0,...,None,No,Steady,No,Steady,Ch,Yes,NO,1,1


In [29]:
# Let's start with handling each column present in the dataframe

In [28]:
print(df['race'].unique())

['Caucasian' 'AfricanAmerican' '?' 'Other' 'Asian' 'Hispanic']


In [31]:
print(df['race'].value_counts())

Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641
Name: race, dtype: int64


In [32]:
# Combine '?' and 'Other' into 'Unknown'
df['race'] = df['race'].replace(['?', 'Other'], 'Unknown')

In [33]:
# Check the new distribution
print(df['race'].value_counts())

Caucasian          76099
AfricanAmerican    19210
Unknown             3779
Hispanic            2037
Asian                641
Name: race, dtype: int64


In [34]:
print(df['gender'].value_counts())

Female             54708
Male               47055
Unknown/Invalid        3
Name: gender, dtype: int64


In [35]:
df['gender'] = df['gender'].replace(['Unknown/Invalid'], 'Female')

In [36]:
print(df['gender'].value_counts())

Female    54711
Male      47055
Name: gender, dtype: int64


In [37]:
print(df['age'].value_counts())

[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)        161
Name: age, dtype: int64


In [38]:
# Remove the '[' and the ')' characters
df['age'] = df['age'].str.replace('[', '', regex=False).str.replace(')', '', regex=False)

In [39]:
print(df['age'].value_counts())

70-80     26068
60-70     22483
50-60     17256
80-90     17197
40-50      9685
30-40      3775
90-100     2793
20-30      1657
10-20       691
0-10        161
Name: age, dtype: int64


In [40]:
# # Create a numeric 'Age_Sort' column based on the start of the range
# # It takes '70-80', splits at '-', and takes the first number (70)
# df['age_sort'] = df['age'].str.split('-').str[0].astype(int)

In [42]:
print(df['time_in_hospital'].value_counts())

3     17756
2     17224
1     14208
4     13924
5      9966
6      7539
7      5859
8      4391
9      3002
10     2342
11     1855
12     1448
13     1210
14     1042
Name: time_in_hospital, dtype: int64


In [45]:
print(df['num_lab_procedures'].unique())

[ 41  59  11  44  51  31  70  73  68  33  47  62  60  55  49  75  45  29
  35  42  66  36  19  64  25  53  52  87  27  37  46  28  48  72  10   2
  65  67  40  54  58  57  43  32  83  34  39  69  38  56  22  96  78  61
  88  50   1  18  82   9  63  24  71  77  81  76  90  93   3 103  13  80
  85  16  15  12  30  23  17  21  79  26   5  95  97  84  14  74 105  86
  98  20   6  94   8 102 100   7  89  91  92   4 101  99 114 113 111 129
 107 108 106 104 109 120 132 121 126 118]


In [46]:
print(df['num_medications'].unique())

[ 1 18 13 16  8 21 12 28 17 11 15 31  2 23 19  7 20 14 10 22  9 27 25  4
 32  6 30 26 24 33  5 39  3 29 61 40 46 41 36 34 35 50 43 42 37 51 38 45
 54 52 49 62 55 47 44 53 48 57 59 56 60 63 58 70 67 64 69 65 68 66 81 79
 75 72 74]


In [47]:
print(df['number_outpatient'].unique())	

[ 0  2  1  5  7  9  3  8  4 12 11  6 20 15 10 13 14 16 21 35 17 29 36 18
 19 27 22 24 42 39 34 26 33 25 23 28 37 38 40]


In [48]:
print(df['number_emergency'].unique())

[ 0  1  2  4  3  9  5  7  6  8 22 25 10 13 42 16 11 28 15 14 18 12 21 20
 19 46 76 37 64 63 54 24 29]


In [49]:
print(df['number_inpatient'].unique())

[ 0  1  2  3  6  5  4  7  8  9 15 10 11 14 12 13 17 16 21 18 19]


In [50]:
df.columns

Index(['race', 'gender', 'age', 'time_in_hospital', 'num_lab_procedures',
       'num_procedures', 'num_medications', 'number_outpatient',
       'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3',
       'number_diagnoses', 'A1Cresult', 'metformin', 'glipizide', 'glyburide',
       'insulin', 'change', 'diabetesMed', 'readmitted', 'admission_type_id',
       'discharge_disposition_id'],
      dtype='object')

In [51]:
print(df['diag_1'].unique())

['250.83' '276' '648' '8' '197' '414' '428' '398' '434' '250.7' '157'
 '518' '999' '410' '682' '402' '737' '572' 'V57' '189' '786' '427' '996'
 '277' '584' '462' '473' '411' '174' '486' '998' '511' '432' '626' '295'
 '196' '250.6' '618' '182' '845' '423' '808' '250.4' '722' '403' '250.11'
 '784' '707' '440' '151' '715' '997' '198' '564' '812' '38' '590' '556'
 '578' '250.32' '433' 'V58' '569' '185' '536' '255' '250.13' '599' '558'
 '574' '491' '560' '244' '250.03' '577' '730' '188' '824' '250.8' '332'
 '562' '291' '296' '510' '401' '263' '438' '70' '250.02' '493' '642' '625'
 '571' '738' '593' '250.42' '807' '456' '446' '575' '250.41' '820' '515'
 '780' '250.22' '995' '235' '250.82' '721' '787' '162' '724' '282' '514'
 'V55' '281' '250.33' '530' '466' '435' '250.12' 'V53' '789' '566' '822'
 '191' '557' '733' '455' '711' '482' '202' '280' '553' '225' '154' '441'
 '250.81' '349' '?' '962' '592' '507' '386' '156' '200' '728' '348' '459'
 '426' '388' '607' '337' '82' '531' '596' '288' '656

In [52]:
# The diag_1, diag_2, and diag_3 columns contain ICD-9 codes (International Classification of Diseases).
# The Strategy: ICD-9 Grouping: The industry standard is to group these hundreds of codes into 9 high-level clinical categories

In [53]:
import numpy as np

def map_icd9(code):
    if pd.isna(code) or code == '?':
        return 'Unknown'
    
    # Handle codes starting with V or E (Standard ICD-9 classifications)
    if str(code).startswith(('V', 'E')):
        return 'Other'
    
    try:
        val = float(code)
    except ValueError:
        return 'Other'

    # Mapping logic based on standard ranges
    if 390 <= val <= 459 or val == 785:
        return 'Circulatory'
    elif 460 <= val <= 519 or val == 786:
        return 'Respiratory'
    elif 520 <= val <= 579 or val == 787:
        return 'Digestive'
    elif str(code).startswith('250'):
        return 'Diabetes'
    elif 580 <= val <= 629 or val == 788:
        return 'Genitourinary'
    elif 140 <= val <= 239:
        return 'Neoplasms'
    elif 710 <= val <= 739:
        return 'Musculoskeletal'
    elif 800 <= val <= 999:
        return 'Injury'
    else:
        return 'Other'

# Apply the mapping to the selected columns
for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].apply(map_icd9)

In [55]:
print(df['diag_1'].unique())
print(df['diag_2'].unique())
print(df['diag_3'].unique())

['Diabetes' 'Other' 'Neoplasms' 'Circulatory' 'Respiratory' 'Injury'
 'Musculoskeletal' 'Digestive' 'Genitourinary' 'Unknown']
['Unknown' 'Diabetes' 'Neoplasms' 'Circulatory' 'Respiratory' 'Other'
 'Injury' 'Musculoskeletal' 'Genitourinary' 'Digestive']
['Unknown' 'Other' 'Circulatory' 'Diabetes' 'Respiratory' 'Injury'
 'Neoplasms' 'Genitourinary' 'Musculoskeletal' 'Digestive']


In [56]:
df.columns

Index(['race', 'gender', 'age', 'time_in_hospital', 'num_lab_procedures',
       'num_procedures', 'num_medications', 'number_outpatient',
       'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3',
       'number_diagnoses', 'A1Cresult', 'metformin', 'glipizide', 'glyburide',
       'insulin', 'change', 'diabetesMed', 'readmitted', 'admission_type_id',
       'discharge_disposition_id'],
      dtype='object')

In [57]:
print(df['number_diagnoses'].value_counts())

9     49474
5     11393
8     10616
7     10393
6     10161
4      5537
3      2835
2      1023
1       219
16       45
10       17
13       16
11       11
15       10
12        9
14        7
Name: number_diagnoses, dtype: int64


In [58]:
print(df['A1Cresult'].value_counts())

None    84748
>8       8216
Norm     4990
>7       3812
Name: A1Cresult, dtype: int64


In [59]:
# 1. Human-Readable Mapping
a1c_labels = {
    '>8': 'High (>8)',
    '>7': 'Elevated (>7)',
    'Norm': 'Normal',
    'None': 'Not Measured'
}

# 2. Define Numeric Mapping for ML (Ordinal Encoding)
a1c_math = {
    'High (>8)': 3,
    'Elevated (>7)': 2,
    'Normal': 1,
    'Not Measured': 0
}

# Apply the label cleanup to the ORIGINAL column
df['A1Cresult'] = df['A1Cresult'].map(a1c_labels)

# Create the numeric column for the model
df['A1Cresult_numeric'] = df['A1Cresult'].map(a1c_math)

# Verification
print(df[['A1Cresult', 'A1Cresult_numeric']].head())

      A1Cresult  A1Cresult_numeric
0  Not Measured                  0
1  Not Measured                  0
2  Not Measured                  0
3  Not Measured                  0
4  Not Measured                  0


In [60]:
print(df['A1Cresult'].value_counts())
print(df['A1Cresult_numeric'].value_counts())

Not Measured     84748
High (>8)         8216
Normal            4990
Elevated (>7)     3812
Name: A1Cresult, dtype: int64
0    84748
3     8216
1     4990
2     3812
Name: A1Cresult_numeric, dtype: int64


In [61]:
# List of columns to check
cols_to_check = [
    'metformin', 'glipizide', 'glyburide', 'insulin', 
    'change', 'diabetesMed', 'readmitted', 
    'admission_type_id', 'discharge_disposition_id'
]

print("--- Value Counts for Clinical & Administrative Columns ---\n")

for col in cols_to_check:
    print(f"Column: {col}")
    print(df[col].value_counts())
    print("-" * 30) # Visual separator

--- Value Counts for Clinical & Administrative Columns ---

Column: metformin
No        81778
Steady    18346
Up         1067
Down        575
Name: metformin, dtype: int64
------------------------------
Column: glipizide
No        89080
Steady    11356
Up          770
Down        560
Name: glipizide, dtype: int64
------------------------------
Column: glyburide
No        91116
Steady     9274
Up          812
Down        564
Name: glyburide, dtype: int64
------------------------------
Column: insulin
No        47383
Steady    30849
Down      12218
Up        11316
Name: insulin, dtype: int64
------------------------------
Column: change
No    54755
Ch    47011
Name: change, dtype: int64
------------------------------
Column: diabetesMed
Yes    78363
No     23403
Name: diabetesMed, dtype: int64
------------------------------
Column: readmitted
NO     54864
>30    35545
<30    11357
Name: readmitted, dtype: int64
------------------------------
Column: admission_type_id
1    53990
3    1886

In [62]:
# Logic: 0 = No Med, 1 = Stable, 2 = Dosage Change (Adjustment)
med_map = {'No': 0, 'Steady': 1, 'Up': 2, 'Down': 2}

for col in ['metformin', 'glipizide', 'glyburide', 'insulin']:
    df[f'{col}_numeric'] = df[col].map(med_map)

In [63]:
df['change'] = df['change'].map({'Ch': 1, 'No': 0})
df['diabetesMed'] = df['diabetesMed'].map({'Yes': 1, 'No': 0})

In [64]:
# 1 = High Risk (Readmitted < 30 days), 0 = Low Risk (Everyone else)
df['readmit_high_risk'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)

In [ ]:
# # Mapping for Admission Type (Standard UCI Mapping)
# admission_map = {1: 'Emergency', 2: 'Urgent', 3: 'Elective', 5: 'Not Available', 6: 'Null'}
# df['admission_type_name'] = df['admission_type_id'].map(admission_map).fillna('Other')

# # IMPORTANT: Remove patients who passed away (Discharge IDs 11, 13, 14, 19, 20, 21)
# death_ids = [11, 13, 14, 19, 20, 21]
# df = df[~df['discharge_disposition_id'].isin(death_ids)]

In [65]:
# 1. Official Mapping from IDS_mapping.csv
admission_map = {
    1: 'Emergency',
    2: 'Urgent',
    3: 'Elective',
    4: 'Newborn',
    5: 'Not Available',
    6: 'NULL',
    7: 'Trauma Center',
    8: 'Not Mapped'
}

# 2. Apply Mapping
df['admission_type_name'] = df['admission_type_id'].map(admission_map).fillna('Other')

# 3. Identify Exclusion IDs (Expired/Hospice)
# Based on the provided CSV:
# 11: Expired
# 13: Hospice / home
# 14: Hospice / medical facility
# 19: Expired at home (hospice)
# 20: Expired in a medical facility (hospice)
# 21: Expired, place unknown (hospice)
exclusion_ids = [11, 13, 14, 19, 20, 21]

# 4. Filter the dataset to remove Survival Bias
original_rows = len(df)
df = df[~df['discharge_disposition_id'].isin(exclusion_ids)]

# 5. Handle "NULL" or "Not Mapped" for Discharge (Optional but recommended)
# ID 18, 25, and 26 are essentially missing data
df['discharge_disposition_id'] = df['discharge_disposition_id'].replace([18, 25, 26], np.nan)

print(f"Data Integrity Check:")
print(f"- Rows removed (Death/Hospice): {original_rows - len(df)}")
print(f"- Final Patient Count for Model: {len(df)}")

Data Integrity Check:
- Rows removed (Death/Hospice): 2423
- Final Patient Count for Model: 99343


C:\Users\PRANAV\AppData\Local\Temp\ipykernel_6156\2199358618.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['discharge_disposition_id'] = df['discharge_disposition_id'].replace([18, 25, 26], np.nan)


In [68]:
df.shape

(99343, 31)

In [69]:
df.columns

Index(['race', 'gender', 'age', 'time_in_hospital', 'num_lab_procedures',
       'num_procedures', 'num_medications', 'number_outpatient',
       'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3',
       'number_diagnoses', 'A1Cresult', 'metformin', 'glipizide', 'glyburide',
       'insulin', 'change', 'diabetesMed', 'readmitted', 'admission_type_id',
       'discharge_disposition_id', 'A1Cresult_numeric', 'metformin_numeric',
       'glipizide_numeric', 'glyburide_numeric', 'insulin_numeric',
       'readmit_high_risk', 'admission_type_name'],
      dtype='object')

In [70]:
# Data Handling

# Logic: A patient with high 'Total_Visits' is statistically much more likely to return.
df['total_visits'] = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']

C:\Users\PRANAV\AppData\Local\Temp\ipykernel_6156\2447229664.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['total_visits'] = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']


In [71]:
# Count how many of the 4 common meds had a change (Up/Down coded as 2 in our previous step)
med_cols_numeric = ['metformin_numeric', 'glipizide_numeric', 'glyburide_numeric', 'insulin_numeric']
df['med_adj_count'] = (df[med_cols_numeric] == 2).sum(axis=1)

C:\Users\PRANAV\AppData\Local\Temp\ipykernel_6156\3599677605.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['med_adj_count'] = (df[med_cols_numeric] == 2).sum(axis=1)


In [72]:
df2=df.copy()

In [73]:
cols_to_drop = ['admission_type_id', 'discharge_disposition_id', 'readmitted']
df.drop(columns=cols_to_drop, inplace=True)

c:\ProgramData\anaconda3\envs\environment\lib\site-packages\pandas\core\frame.py:4906: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return super().drop(


In [74]:
# Ensure categorical columns are 'category' type for efficiency
categorical_cols = ['race', 'gender', 'age', 'diag_1', 'diag_2', 'diag_3', 'admission_type_name']
for col in categorical_cols:
    df[col] = df[col].astype('category')

print(df.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 99343 entries, 0 to 101765
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   race                 99343 non-null  category
 1   gender               99343 non-null  category
 2   age                  99343 non-null  category
 3   time_in_hospital     99343 non-null  int64   
 4   num_lab_procedures   99343 non-null  int64   
 5   num_procedures       99343 non-null  int64   
 6   num_medications      99343 non-null  int64   
 7   number_outpatient    99343 non-null  int64   
 8   number_emergency     99343 non-null  int64   
 9   number_inpatient     99343 non-null  int64   
 10  diag_1               99343 non-null  category
 11  diag_2               99343 non-null  category
 12  diag_3               99343 non-null  category
 13  number_diagnoses     99343 non-null  int64   
 14  A1Cresult            99343 non-null  object  
 15  metformin         

C:\Users\PRANAV\AppData\Local\Temp\ipykernel_6156\705485964.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype('category')


In [75]:
# Convert remaining object columns to category
obj_cols = ['A1Cresult', 'metformin', 'glipizide', 'glyburide', 'insulin']
for col in obj_cols:
    df[col] = df[col].astype('category')

C:\Users\PRANAV\AppData\Local\Temp\ipykernel_6156\3632425306.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype('category')


In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 99343 entries, 0 to 101765
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   race                 99343 non-null  category
 1   gender               99343 non-null  category
 2   age                  99343 non-null  category
 3   time_in_hospital     99343 non-null  int64   
 4   num_lab_procedures   99343 non-null  int64   
 5   num_procedures       99343 non-null  int64   
 6   num_medications      99343 non-null  int64   
 7   number_outpatient    99343 non-null  int64   
 8   number_emergency     99343 non-null  int64   
 9   number_inpatient     99343 non-null  int64   
 10  diag_1               99343 non-null  category
 11  diag_2               99343 non-null  category
 12  diag_3               99343 non-null  category
 13  number_diagnoses     99343 non-null  int64   
 14  A1Cresult            99343 non-null  category
 15  metformin         

In [77]:
# Check for any remaining '?' in the entire dataframe
remaining_q_marks = (df == '?').sum().sum()
print(f"Total '?' remaining in dataframe: {remaining_q_marks}")

# If any are found, see which columns they are in:
if remaining_q_marks > 0:
    print(df.columns[(df == '?').any()].tolist())

Total '?' remaining in dataframe: 0


In [78]:
# Check class imbalance
# Calculate the percentage of High Risk vs Low Risk
risk_counts = df['readmit_high_risk'].value_counts(normalize=True) * 100
print(f"Target Distribution:\n{risk_counts}")

Target Distribution:
0    88.611175
1    11.388825
Name: readmit_high_risk, dtype: float64


SMOTE (Synthetic Minority Over-sampling Technique): Creating "synthetic" versions of the 11% to help the model learn their patterns better.

Class Weighting: Telling the Random Forest that "Missing a High-Risk patient is 8x more expensive than misidentifying a Low-Risk one."

PR-Curve: Using Precision-Recall curves instead of standard ROC curves to measure success.

In [80]:
from pathlib import Path

# Create a folder called 'processed_data' if it doesn't exist
output_dir = Path('processed_data')
output_dir.mkdir(parents=True, exist_ok=True)

# Save the file into that folder
df.to_csv(output_dir / 'diabetes_clean_v1.csv', index=False)

print(f"File saved successfully in: {output_dir.absolute()}")

File saved successfully in: d:\Healthcare Data Analytics Project\notebooks\processed_data


In [3]:
import pandas as pd 
df = pd.read_csv("processed_data/diabetes_clean_v1.csv")

In [4]:
df.isnull().sum()

race                      0
gender                    0
age                       0
time_in_hospital          0
num_lab_procedures        0
num_procedures            0
num_medications           0
number_outpatient         0
number_emergency          0
number_inpatient          0
diag_1                    0
diag_2                    0
diag_3                    0
number_diagnoses          0
A1Cresult                 0
metformin                 0
glipizide                 0
glyburide                 0
insulin                   0
change                    0
diabetesMed               0
A1Cresult_numeric         0
metformin_numeric         0
glipizide_numeric         0
glyburide_numeric         0
insulin_numeric           0
readmit_high_risk         0
admission_type_name    5207
total_visits              0
med_adj_count             0
dtype: int64

In [5]:
df['admission_type_name'] = df['admission_type_name'].fillna('Other/Unknown')

In [7]:
# List of columns that are actually categories
cat_cols = [
    'race', 'gender', 'age', 'diag_1', 'diag_2', 'diag_3', 
    'A1Cresult', 'metformin', 'glipizide', 'glyburide', 'insulin', 
    'admission_type_name'
]

for col in cat_cols:
    df[col] = df[col].astype('category')

# Check info again - you'll see memory usage drop significantly!
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99343 entries, 0 to 99342
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   race                 99343 non-null  category
 1   gender               99343 non-null  category
 2   age                  99343 non-null  category
 3   time_in_hospital     99343 non-null  int64   
 4   num_lab_procedures   99343 non-null  int64   
 5   num_procedures       99343 non-null  int64   
 6   num_medications      99343 non-null  int64   
 7   number_outpatient    99343 non-null  int64   
 8   number_emergency     99343 non-null  int64   
 9   number_inpatient     99343 non-null  int64   
 10  diag_1               99343 non-null  category
 11  diag_2               99343 non-null  category
 12  diag_3               99343 non-null  category
 13  number_diagnoses     99343 non-null  int64   
 14  A1Cresult            99343 non-null  category
 15  metformin          

In [9]:
from pathlib import Path

# Create a folder called 'processed_data' if it doesn't exist
output_dir = Path('processed_data')
output_dir.mkdir(parents=True, exist_ok=True)

# Save the file into that folder
df.to_csv(output_dir / 'cleaned_data.csv', index=False)

print(f"File saved successfully in: {output_dir.absolute()}")

File saved successfully in: d:\Healthcare Data Analytics Project\notebooks\processed_data
